# Link prediction eval results

Traverses a `lp_eval/<method>` directory (files named `dataset-partitions-iteration.out`) and builds a dataframe with columns `dataset`, `partitions`, `iteration`, `Hits@1`, `Hits@3`, `Hits@10`, `MRR`, `F1` parsed from each file.

`F1` is the graph-reconstruction F1 score (`Best achieved F1 score` line) reported for that particular run's partitioning search — it is *not* averaged across runs, it's a single per-run value like the other metrics.

In [1]:
import os
import re
import pandas as pd

LP_EVAL_DIR = os.path.join("..", "lp_eval")

METRIC_PATTERNS = {
    "Hits@1": re.compile(r"Full Model - Hits@1:\s*([\d.]+)"),
    "Hits@3": re.compile(r"Full Model - Hits@3:\s*([\d.]+)"),
    "Hits@10": re.compile(r"Full Model - Hits@10:\s*([\d.]+)"),
    "MRR": re.compile(r"Full Model - MRR:\s*([\d.]+)"),
    # per-run graph-reconstruction F1 (partitioning search), not averaged across runs
    "F1": re.compile(r"Best achieved F1 score:\s*([\d.]+)"),
}

# dataset names can themselves contain dashes (e.g. AS-Oregon, Cit-HepPh),
# so only the trailing `-partitions-iteration` is peeled off.
FILENAME_PATTERN = re.compile(r"^(?P<dataset>.+)-(?P<partitions>\d+)-(?P<iteration>\d+)\.out$")


def parse_result_file(path):
    """Extract Hits@1/3/10, MRR and reconstruction F1 from a single lp_eval .out log file."""
    with open(path, "r") as f:
        content = f.read()
    metrics = {}
    for name, pattern in METRIC_PATTERNS.items():
        match = pattern.search(content)
        metrics[name] = float(match.group(1)) if match else None
    return metrics


def build_results_dataframe(directory):
    """Traverse `directory` for dataset-partitions-iteration.out files and
    parse each into a row of link prediction metrics."""
    rows = []
    for filename in sorted(os.listdir(directory)):
        match = FILENAME_PATTERN.match(filename)
        if not match:
            continue
        metrics = parse_result_file(os.path.join(directory, filename))
        rows.append({
            "dataset": match.group("dataset"),
            "partitions": int(match.group("partitions")),
            "iteration": int(match.group("iteration")),
            **metrics,
        })
    columns = ["dataset", "partitions", "iteration", "Hits@1", "Hits@3", "Hits@10", "MRR", "F1"]
    return pd.DataFrame(rows, columns=columns)

In [2]:
bilinear_dir = os.path.join(LP_EVAL_DIR, "bilinear")
df = build_results_dataframe(bilinear_dir)
df

,dataset,partitions,iteration,Hits@1,Hits@3,Hits@10,MRR,F1
0,AS-Oregon,1,0,0.156,0.290,0.447,0.2552,0.333738
1,AS-Oregon,1,1,0.138,0.286,0.440,0.2390,0.337377
2,AS-Oregon,1,2,0.154,0.275,0.446,0.2449,0.340755
3,AS-Oregon,1,3,0.149,0.293,0.464,0.2503,0.338278
4,AS-Oregon,1,4,0.151,0.280,0.449,0.2478,0.341372
...,...,...,...,...,...,...,...,...
155,Cit-HepPh,8,5,0.264,0.431,0.585,0.3786,0.607252
156,Cit-HepPh,8,6,0.250,0.425,0.612,0.3711,0.625600
157,Cit-HepPh,8,7,0.254,0.429,0.618,0.3759,0.619705
158,Cit-HepPh,8,8,0.234,0.415,0.604,0.3583,0.624652


In [3]:
# Per-metric table: rows = dataset, columns = partition count, cells = "mean% ± std%"
# across the iterations for that (dataset, partitions) group.
METRICS = ["Hits@1", "Hits@3", "Hits@10", "MRR", "F1"]


def build_summary_table(df, metric):
    grouped = df.groupby(["dataset", "partitions"])[metric].agg(["mean", "std"])
    formatted = grouped.apply(lambda row: f"{row['mean'] * 100:.2f}% ± {row['std'] * 100:.2f}%", axis=1)
    table = formatted.unstack("partitions")
    table.columns = [f"P={p}" for p in table.columns]
    return table


summary_tables = {}
for metric in METRICS:
    table = build_summary_table(df, metric)
    summary_tables[metric] = table
    print(f"Metric: {metric}")
    display(table)

Metric: Hits@1


,P=1,P=2,P=4,P=8
dataset,,,,
AS-Oregon,14.99% ± 1.07%,9.70% ± 1.23%,7.83% ± 1.13%,7.57% ± 1.40%
AstroPh,56.79% ± 1.58%,38.54% ± 1.70%,32.57% ± 2.29%,27.68% ± 2.34%
CITESEER,17.70% ± 1.14%,11.08% ± 1.79%,8.08% ± 0.99%,5.76% ± 1.10%
Cit-HepPh,48.38% ± 1.47%,31.05% ± 2.81%,27.10% ± 2.54%,24.90% ± 1.91%


Metric: Hits@3


,P=1,P=2,P=4,P=8
dataset,,,,
AS-Oregon,29.22% ± 1.24%,21.21% ± 1.34%,17.64% ± 1.63%,16.60% ± 2.03%
AstroPh,77.48% ± 1.50%,58.30% ± 2.02%,51.70% ± 2.66%,44.71% ± 3.24%
CITESEER,32.25% ± 2.93%,19.69% ± 1.80%,15.54% ± 1.02%,12.01% ± 1.69%
Cit-HepPh,73.21% ± 1.82%,51.77% ± 4.09%,45.33% ± 4.16%,42.48% ± 2.77%


Metric: Hits@10


,P=1,P=2,P=4,P=8
dataset,,,,
AS-Oregon,45.39% ± 1.44%,36.97% ± 1.19%,32.74% ± 3.00%,31.59% ± 2.28%
AstroPh,89.91% ± 0.78%,74.46% ± 1.37%,69.70% ± 1.91%,63.97% ± 2.55%
CITESEER,47.37% ± 3.50%,30.71% ± 2.55%,24.50% ± 1.34%,20.73% ± 1.73%
Cit-HepPh,89.85% ± 0.56%,72.40% ± 3.12%,64.52% ± 2.83%,60.36% ± 2.20%


Metric: MRR


,P=1,P=2,P=4,P=8
dataset,,,,
AS-Oregon,25.08% ± 0.95%,18.69% ± 1.05%,15.96% ± 1.51%,15.34% ± 1.63%
AstroPh,68.74% ± 1.32%,51.03% ± 1.55%,45.19% ± 2.19%,39.64% ± 2.28%
CITESEER,27.61% ± 1.82%,17.90% ± 1.80%,13.80% ± 0.71%,10.94% ± 1.17%
Cit-HepPh,62.95% ± 1.13%,44.85% ± 3.05%,39.60% ± 2.81%,36.91% ± 2.02%


Metric: F1


,P=1,P=2,P=4,P=8
dataset,,,,
AS-Oregon,33.88% ± 0.28%,38.88% ± 1.91%,45.97% ± 3.79%,43.20% ± 1.78%
AstroPh,70.62% ± 0.08%,67.76% ± 0.09%,67.84% ± 0.12%,67.75% ± 0.11%
CITESEER,35.88% ± 1.46%,43.92% ± 3.25%,49.72% ± 2.34%,55.68% ± 6.12%
Cit-HepPh,54.87% ± 0.06%,57.41% ± 0.79%,60.35% ± 2.12%,61.59% ± 0.87%


In [4]:
# Spearman correlation between reconstruction F1 and MRR, pooled across all
# runs (all datasets/partitions/iterations together, not per-dataset).
from scipy import stats

spearman_corr, spearman_p = stats.spearmanr(df["F1"], df["MRR"])
print(f"Spearman correlation (F1 vs MRR), n={len(df)}: rho={spearman_corr:.4f}, p={spearman_p:.4g}")

Spearman correlation (F1 vs MRR), n=160: rho=0.6189, p=2.719e-18


In [5]:
# Mann-Whitney U test + Cohen's d for MRR, comparing adjacent partition counts
# (P=1 vs P=2, P=2 vs P=4, P=4 vs P=8) within each dataset.
import numpy as np


def cohens_d(a, b):
    pooled_std = np.sqrt((a.std(ddof=1) ** 2 + b.std(ddof=1) ** 2) / 2)
    return (b.mean() - a.mean()) / pooled_std if pooled_std > 0 else np.nan


def metric_values(dataset, partitions, metric):
    mask = (df["dataset"] == dataset) & (df["partitions"] == partitions)
    return df.loc[mask, metric].to_numpy()


partition_levels = sorted(df["partitions"].unique())
adjacent_pairs = list(zip(partition_levels, partition_levels[1:]))

rows = []
for dataset in sorted(df["dataset"].unique()):
    for p_a, p_b in adjacent_pairs:
        a = metric_values(dataset, p_a, "MRR")
        b = metric_values(dataset, p_b, "MRR")
        if len(a) < 2 or len(b) < 2:
            continue
        u_stat, p_u = stats.mannwhitneyu(a, b, alternative="two-sided")
        rows.append({
            "dataset": dataset,
            "comparison": f"P={p_a} vs P={p_b}",
            "n1": len(a),
            "n2": len(b),
            "Mann-Whitney U": u_stat,
            "p-value": p_u,
            "Cohen's d": cohens_d(a, b),
            "significant (α=0.05)": "Yes" if p_u < 0.05 else "No",
        })

mw_mrr_df = pd.DataFrame(rows)
mw_mrr_df

,dataset,comparison,n1,n2,Mann-Whitney U,p-value,Cohen's d,significant (α=0.05)
0,AS-Oregon,P=1 vs P=2,10,10,100.0,0.000183,-6.366286,Yes
1,AS-Oregon,P=2 vs P=4,10,10,96.0,0.000583,-2.103067,Yes
2,AS-Oregon,P=4 vs P=8,10,10,61.0,0.427355,-0.394328,No
3,AstroPh,P=1 vs P=2,10,10,100.0,0.000183,-12.272210,Yes
4,AstroPh,P=2 vs P=4,10,10,99.0,0.000246,-3.067875,Yes
5,AstroPh,P=4 vs P=8,10,10,96.0,0.000583,-2.480379,Yes
6,CITESEER,P=1 vs P=2,10,10,100.0,0.000183,-5.354722,Yes
7,CITESEER,P=2 vs P=4,10,10,99.0,0.000246,-2.984246,Yes
8,CITESEER,P=4 vs P=8,10,10,100.0,0.000183,-2.964108,Yes
9,Cit-HepPh,P=1 vs P=2,10,10,100.0,0.000182,-7.863331,Yes
